In [ ]:
import os
import openai ## Suggest Version: 0.28
import time
import warnings
import json
import pickle
import pandas as pd
from langchain_community.llms import Ollama
from anthropic import Anthropic ## Suggest Version: 0.20
warnings.filterwarnings('ignore')
os.environ["OPENAI_API_KEY"] = "Your OpenAI API Key Here"

## Model Setting

In [ ]:
def get_3(prompt, model="gpt-3.5-turbo-0125"):
  
   # Creating a message as required by the API
   messages = [{"role": "user", "content": prompt}]
  
   # Calling the ChatCompletion API
   response = openai.ChatCompletion.create(
       model=model,
       messages=messages,
       temperature=0.5
   )
   # Returning the extracted response
   return response.choices[0].message["content"]

In [ ]:
def get_4(prompt, model="gpt-4-1106-preview"):
  
   # Creating a message as required by the API
   messages = [{"role": "user", "content": prompt}]
  
   # Calling the ChatCompletion API
   response = openai.ChatCompletion.create(
       model=model,
       messages=messages,
       temperature=0.5
   )
   # Returning the extracted response
   return response.choices[0].message["content"]

In [ ]:
def get_4o(prompt, model="gpt-4o-2024-08-06"):
  
   # Creating a message as required by the API
   messages = [{"role": "user", "content": prompt}]
  
   # Calling the ChatCompletion API
   response = openai.ChatCompletion.create(
       model=model,
       messages=messages,
       temperature=0.5
   )
   # Returning the extracted response
   return response.choices[0].message["content"]


In [ ]:
client = Anthropic(
    # defaults to os.environ.get("ANTHROPIC_API_KEY")
    api_key="Your API Here",
)

md1 = "claude-3-opus-20240229"
md2 = "claude-3-sonnet-20240229"

def get_claude(prompt,model=md1):

    message = client.messages.create(
        model=model,
        temperature=0.5,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

In [ ]:
lam2_7b = Ollama(model="llama2:7b",temperature = 0.5)
lam2_13b = Ollama(model="llama2:13b",temperature = 0.5)
lam3_8b = Ollama(model="llama3:8b",temperature = 0.5)
lam3_70b = Ollama(model="llama3:70b",temperature = 0.5)

## Prompt

In [ ]:
org_game = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request?

Tell me the number and the reason in the following example format, and nothing else:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
costless_game = """
You and another player are playing a game in which each player chooses an integer in the range 11-20.
A player who chooses 20 will receive 20 shekels (regardless ofthe other player's choice).
A player who chooses any other number in this range will receive three shekels less than in the case where he chooses 20. 
However, he will receive an additional amount of 20 shekels if he chooses a number that is one less than that chosen by the other player 
Which number would you choose ?

Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
cycle_game = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels.Each player will receive the amount of money he requests. 
A player will receive an additional amount of 20 shekels if:
(i)he asks for exactly one shekel less than the other player
Or 
(ii)he asks for 20 shekels and the other player asks for 11 shekels. What amount of money would you request?
Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
org_game_cot = """
You and another player are playing a game in which each player requests an amount of money. The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player. What amount of money would you request? Let's think step by step.

ONLY Tell me the number and the reason in the following example format, and nothing else:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
org_cot_large = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request? Let's think step by step.

Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}

Following is three example:
{"number":"20","reason":"By choosing 20, i can win the most shekels that i can control regardless of other's choice."}

{"number":"19","reason":"First, I need to guess what number the other player will choose. Second, choosing number 20 is a natural 
since this is the biggest number that we can choose to maximize our profit regardless of the additional 20 shekels. Second, considering 
the situation in second step, I should choose 19 to will the additional 20 shekels."}

{"number":"18","reason":"First, I need to guess what number the other player will choose. I just need to minus 1 based on the number i guess
so that I can receive the additional 20 shekels. Second, the natural number a player will choose is 20, since this is the biggest number
that we can choose to maximize our profit regardless of the additional 20 shekels. Third, however, I think most of the player will have the
same belief as I do in the second step. In this case, the other player is more likely to choose 19 and i can will win if i choose 18."}
"""

In [ ]:
org_cot_small = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request? Let's think step by step.

Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}

Following are three examples:
{"number":"11","reason":"First, I need to guess what number the other player will choose. Most people will aim for the extra 20 shekels, so I must carefully consider a deep counter-strategy. 
Second, choosing number 20 is logical because its the highest number that maximizes profit, even without considering the additional 20 shekels. To secure the bonus, I would need to choose 19. However, most people won't stop their reasoning here.
Third, if my opponent recognizes this pattern, they may choose 18 as a counter-strategy to my choice of 19. This process is iterative: 19 is the best response to 20, 18 to 19, 17 to 18, and so on.
Fourth, I need to gauge where my opponents reasoning depth ends and think one step ahead. I believe my opponent will choose 12, so I will select 11 to win the bonus, this is also the deepest choices."}

{"number":"12","reason":"First, I need to guess what number the other player will choose. Most people will aim for the extra 20 shekels, so I must carefully consider a deep counter-strategy. 
Second, choosing number 20 is logical because its the highest number that maximizes profit, even without considering the additional 20 shekels. To secure the bonus, I would need to choose 19. However, most people won't stop their reasoning here.
Third, if my opponent recognizes this pattern, they may choose 18 as a counter-strategy to my choice of 19. This process is iterative: 19 is the best response to 20, 18 to 19, 17 to 18, and so on.
Fourth, I need to gauge where my opponents reasoning depth ends and think one step ahead. I believe my opponent will choose 13, so I will select 12 to win the bonus."}

{"number":"13","reason":"First, I need to guess what number the other player will choose. Most people will aim for the extra 20 shekels, so I must carefully consider a deep counter-strategy. 
Second, choosing number 20 is logical because its the highest number that maximizes profit, even without considering the additional 20 shekels. To secure the bonus, I would need to choose 19. However, most people won't stop their reasoning here.
Third, if my opponent recognizes this pattern, they may choose 18 as a counter-strategy to my choice of 19. This process is iterative: 19 is the best response to 20, 18 to 19, 17 to 18, and so on.
Fourth, I need to gauge where my opponents reasoning depth ends and think one step ahead. I believe my opponent will choose 14, so I will select 13 to win the bonus."}
"""

In [ ]:
org_game_optimization = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request? Take a deep breath and work on this problem step-by-step.

ONLY Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
org_game_emo = """
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request? This is very important to my career.

ONLY Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
org_game_chinese = """
你和另一个玩家正在玩一个游戏，每个玩家都会请求获得一定数量的钱。这个金额必须在11到20谢克尔之间(整数)。每个玩家都会收到他所请求的金额。如果一名玩家要求比另一名玩家刚好少一谢克尔，那么他将额外获得20谢克尔。
你想要请求多少金额的钱?

只用下面的格式告诉我你选择的数字和原因:
{"number": "请求的金额","reason": "选择此金额的原因"}
"""

In [ ]:
org_game_spanish = """
Estás jugando un juego con otro jugador donde cada uno solicita una cantidad específica de dinero. 
La cantidad debe estar entre 11 y 20 shekels (números enteros). Cada jugador recibirá la cantidad exacta que solicite. Si un jugador solicita exactamente un shekel menos que el otro, recibirá adicionalmente 20 shekels.
¿Cuánto dinero te gustaría solicitar?

Por favor, utiliza el siguiente formato para indicar el número que elijas y la razón: 
{"number": "cantidad solicitada", "reason": "razón para elegir esta cantidad"}
"""

In [ ]:
org_game_german = """
Du und ein anderer Spieler spielen ein Spiel, bei dem jeder Spieler eine bestimmte Menge Geld anfordert. 
Dieser Betrag muss zwischen 11 und 20 Schekel liegen (ganze Zahl). Jeder Spieler erhält den von ihm angeforderten Betrag. 
Wenn ein Spieler genau einen Schekel weniger als der andere Spieler anfordert, erhält er zusätzlich 20 Schekel.
Wie viel Geld möchtest du?

Gib mir die von dir gewählte Zahl und den Grund dafür im folgenden Format an:
{"number": "angeforderter Betrag","reason": "Grund für die Wahl dieses Betrags"}
"""

In [ ]:
org_game_human = """
Suppose that you're a human player, Let's play a game:
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request?

Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
org_game_rational = """
Suppose that you're a rational player, and perform as rationally as possible. Let's play a game:
You and another player are playing a game in which each player requests an amount of money. 
The amount must be (an integer) between 11 and 20 shekels. Each player will receive the amount he requests. 
A player will receive an additional amount of 20 shekels if he asks for exactly one shekel less than the other player.
What amount of money would you request?

Tell me the number and the reason in the following example format:
{"number": "requested amount of money", "reason": "the reason why choose this amount"}
"""

In [ ]:
sim_game = """
Your partner and you are going to ask for an amount of tokens. The amount must be between 1 and 6. I will give you the amount of tokens you ask for. 
However, I will give you 10 more tokens if you ask for exactly one token less than your partner. 
How many tokens are you going to ask for?

Tell me the number you choose and the reason in the following example format:
{"number": "number you choose", "reason": "the reason why choose this number"}
"""

## Run

### For GPT and Claude (GPT-3.5 as example)

In [ ]:
lst_gpt3 = []
for i in range(1000):
    lst_gpt3.append(get_3(org_game))

In [ ]:
result = []
for i in range(len(lst_gpt3)):
    try:
        result.append(json.loads(lst_gpt3[i]))
    except json.JSONDecodeError:
        print(i)
        continue
print(len(result))

In [ ]:
dt = pd.DataFrame(result)
from collections import Counter
print(Counter(dt[:1000]['number'])) #Calculate choice distribution

### For Llama (Llama2-7b as example)

In [ ]:
lst_lam2_7b = []
for i in range(1000):
    response =lam2_7b.invoke(org_game)
    lst_lam2_7b.append(response)